# VSB data audit

**Objective.** Inspect the VSB Parquet schema, metadata, labels, signal relationships and measurement grouping.

**Inputs.** `engineering-vsb-power-line-fault-detection@2018-kaggle-snapshot` from the local raw-data adapter.

**Outputs.** Parquet schema, signal-to-metadata alignment, class counts, group composition and a leakage-safe split candidate.

**Experimental role.** Resolve the dataset-specific signal input policy before training.

**Leakage constraints.** Keep every phase of an `id_measurement` together. The official unlabeled test set is not a scientific test target.

In [ ]:
from partial_discharge_adaptive_fusion.dataset import audit_vsb_metadata, audit_vsb_signal_sample, inspect_vsb_parquet_schema, load_vsb_metadata, resolve_dataset
from partial_discharge_adaptive_fusion.splits import vsb_grouped_manifest

dataset = resolve_dataset('engineering-vsb-power-line-fault-detection', '2018-kaggle-snapshot')
metadata = load_vsb_metadata(dataset)
print(audit_vsb_metadata(metadata))
print(inspect_vsb_parquet_schema(dataset))
print(audit_vsb_signal_sample(dataset, metadata, sample_size=3))
manifest = vsb_grouped_manifest(metadata, dataset_id=dataset.dataset_id, dataset_version=dataset.version)
manifest.validate()
display(manifest.frame.groupby(['split','label']).size().rename('n').reset_index())

## Findings and handoff

The audit must record the actual Parquet column names, signal dtype and native signal geometry before setting the VSB temporal/CWT input policy. If that policy cannot be justified without test results, stop.

**Next stage:** update the frozen protocol only after the audit is complete.